<a href="https://colab.research.google.com/github/hanauert/Hausarbeit-GenAI/blob/main/04_stance_detection_gemma3_4b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Load gold annotated dataset**

In [1]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import matthews_corrcoef

In [2]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
subsample_gold = pd.read_csv('/content/drive/MyDrive/FKM/stance_detection/gold_annotated.csv')

In [4]:
# Mapping dictionary
label_map = {
    1.0: "positiv",
    2.0: "negativ",
    3.0: "neutral"
}

# Apply the mapping
subsample_gold['gold_standard'] = subsample_gold['gold_standard'].map(label_map)

In [5]:
subsample_gold['lead'] = subsample_gold['body_text'].str.split('\n\n|\n').str[0]

#**Load Gemma3:4b**

In [6]:
!curl https://ollama.ai/install.sh | sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  35979      0 --:--:-- --:--:-- --:--:-- 35991
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [7]:
import subprocess

# Start Ollama service in the background
ollama_process = subprocess.Popen(["ollama", "serve"])

# Optional: check if it's running
print("Ollama server started in the background.")

Ollama server started in the background.


In [8]:
!curl http://localhost:11434

curl: (7) Failed to connect to localhost port 11434 after 0 ms: Connection refused


In [9]:
!pip install -q ollama openai
import pandas as pd
import ollama
from ollama import chat

In [10]:
ollama.pull("gemma3:4b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

##**Define Metrics**

In [11]:
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score, precision_score, recall_score
import pandas as pd

def evaluate_model(true, pred):
    return {
        'Cohens Kappa': cohen_kappa_score(true, pred),
        'F1': f1_score(true, pred, average='weighted'),
        'Accuracy': accuracy_score(true, pred),
        'Precision': precision_score(true, pred, average='weighted'),
        'Recall': recall_score(true, pred, average='weighted')
    }

##**Annotate with Gemma3:4b - Prompt 2**

In [12]:
from ollama import chat
from pydantic import BaseModel
from typing import List, Literal
import json

class MigrationAnnotation(BaseModel):
   annotation: Literal['positiv', 'negativ', 'neutral']


def classify_with_gemma_structured_p2(text):
    messages = [
    {
        "role": "system",
        "content": (
            "Stance Detection: Klassifiziere den folgenden Absatz.\n"
            "positiv: Der Text zeigt eine befürwortende Haltung gegenüber Migration.\n"
            "negativ: Migration wird im Text eher kritisch betrachtet.\n"
            "neutral: Der Text drückt keine eindeutige Meinung zu Migration aus."
        )
    },
    {
        "role": "user",
        "content": f"{text} --- Wie ist die Haltung des Autors zu Migration? "
    }
]

    response = chat(
        messages=messages,
        model="gemma3:4b",
        format=MigrationAnnotation.model_json_schema(),
        options=dict(seed=42, temperature=0.0)
    )

    return response['message']['content']

###**Annotate Title + Paragraph**

In [13]:
# Apply the classification function to the dataframe
subsample_gold['label_zs_gemma_p2'] = (subsample_gold['title'] + "\n\n" + subsample_gold['paragraph']).apply(classify_with_gemma_structured_p2)

subsample_gold['label_zs_gemma_p2'] = subsample_gold['label_zs_gemma_p2'].str.extract(r'"annotation"\s*:\s*"(\w+)"')

# over 6min computational time with T4

In [14]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard'], subsample_gold['label_zs_gemma_p2']))

              precision    recall  f1-score   support

     negativ       0.33      0.85      0.48        62
     neutral       0.55      0.28      0.38       109
     positiv       0.82      0.35      0.50        79

    accuracy                           0.45       250
   macro avg       0.57      0.50      0.45       250
weighted avg       0.58      0.45      0.44       250



In [15]:
metrics_gemma_p2 = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_zs_gemma_p2'])

print("Evaluation Metrics: Gemma3:4b - Prompt 2")
for metric, value in metrics_gemma_p2.items():
    print(f"{metric}: {value:.3f}")

Evaluation Metrics: Gemma3:4b - Prompt 2
Cohens Kappa: 0.212
F1: 0.439
Accuracy: 0.448
Precision: 0.584
Recall: 0.448


##**Annotate with Gemma3:4b - Prompt 2.2**

In [16]:
from ollama import chat
from pydantic import BaseModel
from typing import List, Literal
import json

class MigrationAnnotation(BaseModel):
   annotation: Literal['positiv', 'negativ', 'neutral']


def classify_with_gemma_structured_p22(text):
    messages = [
    {
        "role": "system",
        "content": (
            "Stance Detection: Klassifiziere den folgenden Absatz.\n"
            "positiv: Der Text zeigt eine eher positive Haltung gegenüber Migrantinnen und Migranten.\n"
            "negativ: Der Text zeigt eine eher negative Haltung gegenüber Migrantinnen und Migranten.\n"
            "neutral: Der Text drückt eine neutrale oder differenzierte Haltung gegenüber Migrantinnen und Migranten aus."
        )
    },
    {
        "role": "user",
        "content": f"{text} --- Wie ist die Haltung des Autors zu Migration? Think."
    }
]

    response = chat(
        messages=messages,
        model="gemma3:4b",
        format=MigrationAnnotation.model_json_schema(),
        options=dict(seed=42, temperature=0.0)
    )

    return response['message']['content']

###**Annotate Title + Paragraph**

In [17]:
# Apply the classification function to the dataframe
subsample_gold['label_zs_gemma_p22'] = (subsample_gold['title'] + "\n\n" + subsample_gold['paragraph']).apply(classify_with_gemma_structured_p22)

subsample_gold['label_zs_gemma_p22'] = subsample_gold['label_zs_gemma_p22'].str.extract(r'"annotation"\s*:\s*"(\w+)"')

In [18]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard'], subsample_gold['label_zs_gemma_p22']))

              precision    recall  f1-score   support

     negativ       0.39      0.76      0.51        62
     neutral       0.62      0.46      0.53       109
     positiv       0.75      0.46      0.57        79

    accuracy                           0.53       250
   macro avg       0.59      0.56      0.54       250
weighted avg       0.60      0.53      0.54       250



In [19]:
metrics_gemma_p22 = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_zs_gemma_p22'])

print("Evaluation Metrics: Gemma3:4b - Prompt 2.2")
for metric, value in metrics_gemma_p22.items():
    print(f"{metric}: {value:.3f}")

Evaluation Metrics: Gemma3:4b - Prompt 2.2
Cohens Kappa: 0.310
F1: 0.536
Accuracy: 0.532
Precision: 0.602
Recall: 0.532


##**Add labels to label dataframe**

In [20]:
labels_df = pd.read_csv('/content/drive/MyDrive/FKM/stance_detection/classification_results/results_NLI_classification.csv')

In [21]:

labels_df['label_gemma3_zs_p22'] = subsample_gold['label_zs_gemma_p22']

In [22]:
labels_df.head()

,date,title,paragraph,newspaper,gold_standard,label_ml_p6,score_ml_p6,label_mlb_p8,score_mlb_p8,label_ls_p6,score_ls_p6,label_llama31_zs_p1,label_gemma3_zs_p22
0,2025-01-02,Wir verschenken zu viel Zeit,Das gilt übrigens auch für die demokratischen ...,FR,positiv,positiv,0.440892,positiv,0.999996,positiv,0.987128,positiv,negativ
1,2024-12-26,Das Armutsrisiko in Deutschland ist rückläufig...,Trotz des gewachsenen Migrantenanteils in der ...,Welt,neutral,neutral,0.625038,neutral,0.988387,positiv,0.415422,positiv,neutral
2,2023-11-26,Arbeitgeber erwarten Wohlstandsverlust durch P...,Die Unionsfraktion hatte stattdessen vorgeschl...,Welt,negativ,negativ,0.933522,negativ,0.999986,negativ,0.999362,neutral,negativ
3,2024-11-25,Das Ende einer Erfolgsstory; Kanzler Scholz fe...,Hauptgründe für die Aufwärtsbewegung sind die ...,Welt,positiv,positiv,0.411763,positiv,0.905089,negativ,0.996768,neutral,negativ
4,2024-09-10,„Investitionen entscheiden über die Leistungsf...,Die Aufnahme einer Erwerbsarbeit ist die beste...,FR,positiv,negativ,0.922741,positiv,0.999997,positiv,0.995294,positiv,negativ


In [23]:
labels_df.to_csv('/content/drive/MyDrive/FKM/stance_detection/classification_results/results_NLI_classification.csv', index=False)